# XAI — Coffee Bean Quality Detection
### Verifikasi "cara berpikir" seluruh 10 model, sebelum peringkat macro-F1 dipercaya penuh

Rancangan lengkap & alasan pemilihan tiap metode ada di
[`docs/xai-strategy.md`](https://github.com/Ardiyanto24/coffee-bean-quality-detection/blob/main/docs/xai-strategy.md).
Ringkasannya: 4-fold CV (langkah berikutnya yang tertunda) cuma menguji stabilitas skor
terhadap pembagian data — bukan apakah model benar secara sebab-akibat. Notebook ini
menjawab itu dulu, untuk **seluruh 10 model** dari
[`CBQD - Training.ipynb`](https://github.com/Ardiyanto24/coffee-bean-quality-detection/blob/main/notebook/CBQD%20-%20Training.ipynb)
(bukan cuma `09_noise_robust` si juara), lewat 5 sudut pandang:

| # | Sudut pandang | Metode |
|---|---|---|
| 1 | Spasial | Grad-CAM, Integrated Gradients, Occlusion (Captum) |
| 2 | Konsep semantik | TCAV — concept split dari fitur hand-crafted EDA (`center_offset`, dll) |
| 3 | Kausal/counterfactual | Probe terprogram: reposisi bean dalam frame, occlusion region |
| 4 | Data lineage | Nearest-neighbor di embedding space |
| 5 | Agregat/statistik | Korelasi tiap temuan di atas terhadap metadata EDA, ≥N sampel/kelas |

**5 keluarga arsitektur** (metode di atas beda adapter per keluarga — lihat `xai-strategy.md` §3-4):
A. CNN polos (02,03,04,05,09) — Grad-CAM native. B. Transformer (06 DeiT) — IG+Occlusion saja
(bukan Grad-CAM). C. Tabular/tree (01 Gradient Boosting) — SHAP, bukan heatmap piksel; TCAV
dilewati (redundan). D. Multi-submodel (07,08) — pipeline A per sub-model/head. E. Ensemble
(10) — gabungan C+A berdampingan.

**5 hipotesis kecurigaan yang diuji** (bukan cuma satu — lima sudut pandang di atas diarahkan
ke lima target berbeda):

| # | Hipotesis | Diuji di | Metode utama |
|---|---|---|---|
| 1 | Posisi bean sebagai shortcut framing (EDA v2 §06: `premium` lebih terpusat) | Section 7, 8, 10 | TCAV `off_center`, probe reposisi |
| 2 | Konsep "rusak" bocor ke kelas non-defect (EDA v3: defect = campuran jenis lain) | Section 12 (baru) | CAV defect-vs-nondefect, uji lintas kelas |
| 3 | Model tidak generalisasi ke kondisi pengambilan gambar `real_world/` yang berbeda | Section 13 (baru) | Distribusi confidence & prediksi, Grad-CAM kualitatif |
| 4 | Warna sebagai artefak sesi pemotretan, bukan ciri bean asli (effect size EDA terbesar) | Section 7, 8, 10 | TCAV `dark_color`, probe color-jitter |
| 5 | Ukuran-di-frame (`area_frac`) sebagai shortcut terpisah dari posisi/bentuk | Section 7, 8, 10 | TCAV `large_area` |

**Checkpoint model:** run training sebelumnya TIDAK menyimpan bobot model (cuma metrik) —
notebook ini melatih ulang ke-10 model dengan kode & seed yang identik (`CBQD -
Training.ipynb`), lalu menyimpan checkpoint-nya ke `models/checkpoints/` dan push ke R2/DVC,
supaya iterasi debug XAI berikutnya TIDAK perlu melatih ulang (cek checkpoint dulu sebelum
melatih).

**PENTING — DRY_RUN:** sama seperti notebook training — `DRY_RUN=True` (epoch kecil + sampel
agregat kecil) dulu untuk memastikan seluruh kode jalan tanpa error, baru `DRY_RUN=False`
untuk run penuh.

**Catatan GPU:** TIDAK pip install/upgrade `torch`/`torchvision`.

## Section 1 — Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)

!pip install -q lightgbm timm captum shap

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

# Lokasi mount dataset di /kaggle/input pernah berubah antar kernel -- cari lewat rglob
# supaya tidak bergantung pada satu struktur path yang diasumsikan.
_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest dari R2 (dvc pull); cek checkpoint model yang mungkin sudah ada

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

CKPT_DIR = Path("models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
existing_ckpts = list(CKPT_DIR.glob("*"))
print(f"Checkpoint model yang sudah ada (dari dvc pull): {len(existing_ckpts)} file")
for p in existing_ckpts:
    print(" -", p.name)

## Section 2 — Konfigurasi

`DRY_RUN=True` -> epoch retrain kecil + sampel agregat kecil (tes kode). Ubah ke `False`
untuk run penuh SETELAH dry-run terbukti jalan bersih dari awal sampai akhir.

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + turunan epoch/patience/ukuran sampel agregat

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = False  # <-- v11 dry-run: isi tabel xai_summary.csv sudah diverifikasi benar
                 # (07/08 TCAV lengkap 4 konsep, H2/H3 lengkap 10 model) -- full run lagi.

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
    SCHEDULER_PATIENCE = 1
    N_AGG_PER_CLASS = 3     # sampel agregat per kelas untuk probe/lineage/TCAV
    N_QUAL = 4              # sampel kualitatif untuk visualisasi heatmap
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45
    EARLY_STOP_PATIENCE = 10
    SCHEDULER_PATIENCE = 4
    N_AGG_PER_CLASS = 999999  # efektif: pakai semua test set (231 gambar)
    N_QUAL = 8

BATCH_SIZE = 32
IMG_SIZE = 224
LR_PHASE1 = 1e-3
LR_PHASE2 = 3e-4
LR_PHASE2_VIT = 5e-5
WEIGHT_DECAY = 0.01
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | epochs phase1/phase2={EPOCHS_PHASE1}/{EPOCHS_PHASE2} | N_AGG_PER_CLASS={N_AGG_PER_CLASS}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    # Kaggle's default image ships a PyTorch build without Pascal (sm_60) kernels,
    # so a P100 allocation crashes deep into training instead of failing fast.
    # kernel-metadata.json pins machine_shape=NvidiaTeslaT4 to avoid this, but
    # assert here too in case that pin is ever dropped or ignored.
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )

## Section 3 — Data: Manifest, Dataset, Transform, DataLoader

Identik dengan `CBQD - Training.ipynb` Section 3 — protokol split & augmentasi yang sama,
supaya model yang dilatih ulang di sini konsisten dengan hasil yang sudah dilaporkan.

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test DataFrame

import pandas as pd

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}")
print(fit_df["label"].value_counts())

In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform (augmentasi rentang untuk train, resize+normalize untuk val/test)

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Versi tanpa normalize -- dipakai probe kausal & visualisasi heatmap (butuh piksel 0-1 mentah)
raw_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])


class BeanDataset(Dataset):
    def __init__(self, df, root_dir, transform, weights=None, label_fn=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        label_fn = label_fn if label_fn is not None else (lambda l: LABEL_TO_IDX[l])
        self.labels = [label_fn(l) for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


fit_ds = BeanDataset(fit_df, PREP_DIR, train_transform)
val_ds = BeanDataset(val_df, PREP_DIR, eval_transform)
test_ds = BeanDataset(test_df, PREP_DIR, eval_transform)

fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("DataLoaders siap.")

## Section 4 — Fungsi Utilitas Model (identik `CBQD - Training.ipynb`)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() & evaluate_combined(): dibutuhkan train_one_model() untuk early stopping

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device, combine_fn=None, class_names=None):
    class_names = class_names if class_names is not None else CLASS_NAMES
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = combine_fn(outputs) if combine_fn is not None else outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds if isinstance(preds, list) else preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}


TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}

In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() & train_one_model(): loop 2-fase + early stopping

import copy
import time


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


def train_one_model(model, head_module, fit_loader, val_loader, device, model_name,
                     lr_phase2=LR_PHASE2, criterion=None, combine_fn=None, class_names=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0
    t0 = time.time()

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY
    )
    for epoch in range(EPOCHS_PHASE1):
        train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device, combine_fn=combine_fn, class_names=class_names)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device, combine_fn=combine_fn, class_names=class_names)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed = time.time() - t0
    print(f"[{model_name}] selesai dilatih ulang dalam {elapsed/60:.1f} menit, val_macro_f1={best_val_f1:.4f}")
    return model, best_val_f1

In [ ]:
# Sub-Step 4.3
# Tujuan: build_model(): factory arsitektur torchvision/timm -> (model, head_module)

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "mobilenet_v3_large":
        m = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "resnet18":
        m = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, num_classes)
        head = m.fc
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "deit_tiny":
        import timm
        m = timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=num_classes)
        head = m.get_classifier()
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)

In [ ]:
# Sub-Step 4.4
# Tujuan: handcrafted_features(): 11 fitur EDA dari gambar mentah -- dipakai Model 01, TCAV concept split, & probe kausal

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    else:
        area_frac = bbox_ratio = center_offset = np.nan
        bbox = (0, 0, w, h)

    edges = cv2.Canny(gray, 100, 200)
    feats = {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }
    return feats, bbox


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p)[0] for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


X_fit, y_fit = build_feature_matrix(fit_df)
X_val, y_val = build_feature_matrix(val_df)
X_test, y_test = build_feature_matrix(test_df)
print("Feature matrix shapes:", X_fit.shape, X_val.shape, X_test.shape)

## Section 5 — Retrain (atau Muat) 10 Model, Simpan Checkpoint

Run `CBQD - Training.ipynb` tidak menyimpan bobot model. Tiap model di sini dicek dulu:
kalau checkpoint sudah ada di `models/checkpoints/` (dari `dvc pull` di Section 1, artinya
sudah pernah dilatih di run sebelumnya), langsung dimuat -- kalau belum, dilatih ulang dengan
kode & seed identik `CBQD - Training.ipynb`, lalu disimpan. Ini membuat iterasi debug bagian
XAI (Section 6 dst) tidak perlu melatih ulang tiap kali notebook di-push ulang.

In [ ]:
# Sub-Step 5.1
# Tujuan: Helper get_or_train_cnn(): cek checkpoint, muat atau latih ulang

retrained_any = False


def get_or_train_cnn(name, arch, num_classes, fit_loader_, val_loader_,
                      lr_phase2=LR_PHASE2, criterion=None, class_names=None):
    global retrained_any
    path = CKPT_DIR / f"{name}.pt"
    model, head = build_model(arch, num_classes)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device))
        model = model.to(device).eval()
        print(f"[{name}] checkpoint dimuat dari {path}")
        return model
    model, val_f1 = train_one_model(model, head, fit_loader_, val_loader_, device, name,
                                     lr_phase2=lr_phase2, criterion=criterion, class_names=class_names)
    torch.save(model.state_dict(), path)
    retrained_any = True
    return model.eval()

In [ ]:
# Sub-Step 5.2
# Tujuan: Model 01 — Gradient Boosting: latih/muat, simpan sebagai .joblib

import joblib


def get_or_train_gb():
    global retrained_any
    path = CKPT_DIR / "01_gradient_boosting.joblib"
    if path.exists():
        model = joblib.load(path)
        print(f"[01_gradient_boosting] checkpoint dimuat dari {path}")
        return model
    import lightgbm as lgb
    model = lgb.LGBMClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                                random_state=SEED, verbose=-1)
    model.fit(X_fit, y_fit)
    joblib.dump(model, path)
    retrained_any = True
    return model


gb_model = get_or_train_gb()
gb_val_f1 = f1_score(y_val, gb_model.predict(X_val), average="macro")
print("01_gradient_boosting val macro-F1:", round(gb_val_f1, 4))

In [ ]:
# Sub-Step 5.3
# Tujuan: Model 02-06 — CNN/ViT: latih/muat lewat loop

_cnn_specs = [
    ("02_mobilenet_v3_large", "mobilenet_v3_large", "LR_PHASE2"),
    ("03_efficientnet_b0", "efficientnet_b0", "LR_PHASE2"),
    ("04_resnet18", "resnet18", "LR_PHASE2"),
    ("05_convnext_tiny", "convnext_tiny", "LR_PHASE2"),
    ("06_deit_tiny", "deit_tiny", "LR_PHASE2_VIT"),
]
cnn_models = {}
for name, arch, lr2_name in _cnn_specs:
    lr2 = LR_PHASE2 if lr2_name == "LR_PHASE2" else LR_PHASE2_VIT
    cnn_models[name] = get_or_train_cnn(name, arch, 4, fit_loader, val_loader, lr_phase2=lr2)
    torch.cuda.empty_cache()
print("Model CNN/ViT siap:", list(cnn_models.keys()))

In [ ]:
# Sub-Step 5.4
# Tujuan: Model 07 — Hierarchical: latih/muat 2 sub-model (tipe 3-kelas, rusak biner)

TYPE_LABEL_FN = lambda l: {"premium": 0, "peaberry": 1, "longberry": 2}[l]
DAMAGE_LABEL_FN = lambda l: 1 if l == "defect" else 0

fit_df_type = fit_df[fit_df["label"] != "defect"].reset_index(drop=True)
val_df_type = val_df[val_df["label"] != "defect"].reset_index(drop=True)

fit_type_loader = DataLoader(BeanDataset(fit_df_type, PREP_DIR, train_transform, label_fn=TYPE_LABEL_FN),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_type_loader = DataLoader(BeanDataset(val_df_type, PREP_DIR, eval_transform, label_fn=TYPE_LABEL_FN),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
fit_damage_loader = DataLoader(BeanDataset(fit_df, PREP_DIR, train_transform, label_fn=DAMAGE_LABEL_FN),
                                batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_damage_loader = DataLoader(BeanDataset(val_df, PREP_DIR, eval_transform, label_fn=DAMAGE_LABEL_FN),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

type_model = get_or_train_cnn("07a_type", "efficientnet_b0", 3, fit_type_loader, val_type_loader,
                               class_names=["premium", "peaberry", "longberry"])
damage_model = get_or_train_cnn("07b_damage", "efficientnet_b0", 2, fit_damage_loader, val_damage_loader,
                                 class_names=["intact", "defect"])
torch.cuda.empty_cache()


def hierarchical_predict_fn(images):
    damage_pred = damage_model(images).argmax(dim=1).cpu().numpy()
    type_pred = type_model(images).argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

In [ ]:
# Sub-Step 5.5
# Tujuan: Model 08 — Multi-task: latih/muat 1 backbone + 2 head

def get_or_train_multitask():
    global retrained_any
    path = CKPT_DIR / "08_multitask.pt"
    model = EfficientNetMultiTask()
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"[08_multitask] checkpoint dimuat dari {path}")
        return model.to(device).eval()

    fit_mt_loader = DataLoader(MultiTaskDataset(fit_df, PREP_DIR, train_transform),
                                batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_mt_loader = DataLoader(MultiTaskDataset(val_df, PREP_DIR, eval_transform),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    @torch.no_grad()
    def _mt_val_f1(m, loader):
        m.eval(); preds, labels_flat = [], []
        for images, damage_labels, type_labels in loader:
            images = images.to(device)
            out_damage, out_type = m(images)
            damage_pred = out_damage.argmax(dim=1).cpu().numpy()
            type_pred = out_type.argmax(dim=1).cpu().numpy()
            combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
            preds.extend(list(combined))
            dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
            true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
            labels_flat.extend(true_flat.tolist())
        return f1_score(labels_flat, preds, average="macro", zero_division=0)

    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_mt_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=LR_PHASE1, weight_decay=WEIGHT_DECAY,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_mt_loader)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_mt_loader)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), path)
    retrained_any = True
    print(f"[08_multitask] selesai dilatih ulang, val_macro_f1={best_val_f1:.4f}")
    return model.eval()


mt_model = get_or_train_multitask()
torch.cuda.empty_cache()


def multitask_predict_fn(images):
    out_damage, out_type = mt_model(images)
    damage_pred = out_damage.argmax(dim=1).cpu().numpy()
    type_pred = out_type.argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

In [ ]:
# Sub-Step 5.6
# Tujuan: Model 09 — Noise-Robust: hitung ulang mistake_score, latih/muat EfficientNet-B0 berbobot

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
rf_noise = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
proba_oof = cross_val_predict(rf_noise, X_fit, y_fit, cv=sgkf_noise,
                               groups=fit_df["cluster_id"].values, method="predict_proba")
true_proba = proba_oof[np.arange(len(y_fit)), y_fit]
max_proba = proba_oof.max(axis=1)
mistake_score = max_proba - true_proba
flagged_mask = mistake_score > 0.5
sample_weights_fit = np.where(flagged_mask, 0.5, 1.0)
print(f"Kandidat mislabel di fit pool: {flagged_mask.sum()} / {len(fit_df)}")

fit_ds_m9 = BeanDataset(fit_df, PREP_DIR, train_transform, weights=sample_weights_fit.tolist())
fit_loader_m9 = DataLoader(fit_ds_m9, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
criterion_m9 = nn.CrossEntropyLoss(label_smoothing=0.1, reduction="none")

noise_robust_model = get_or_train_cnn("09_noise_robust", "efficientnet_b0", 4, fit_loader_m9, val_loader,
                                       criterion=criterion_m9)
cnn_models["09_noise_robust"] = noise_robust_model
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 5.7
# Tujuan: Push checkpoint ke R2/DVC kalau ada yang baru dilatih

print(f"Ada model yang baru dilatih ulang di sesi ini: {retrained_any}")
if retrained_any:
    os.system("dvc add models/checkpoints")
    os.system("dvc push")
    dvc_file = Path("models/checkpoints.dvc")
    if dvc_file.exists():
        print("--- Isi models/checkpoints.dvc (salin ke repo lokal, lalu commit) ---")
        print(dvc_file.read_text())
else:
    print("Semua checkpoint sudah ada dari dvc pull -- tidak ada yang perlu di-push ulang.")

## Section 6 — Fungsi Utilitas XAI Bersama (dipakai keluarga A/B/D)

**Catatan implementasi TCAV:** dibuat manual (bukan `captum.concept.TCAV`) -- proyeksi konsep
lewat logistic regression di layer activation + directional derivative gradien target-class
adalah inti matematis TCAV yang sama, tapi versi manual ini lebih mudah diverifikasi baris per
baris dibanding wiring API `captum.concept.TCAV` yang tidak bisa diuji dulu secara lokal
(tidak ada GPU lokal untuk validasi sebelum push ke Kaggle). Grad-CAM/IG/Occlusion tetap
memakai Captum langsung -- API-nya jauh lebih sederhana (satu pemanggilan `.attribute()`).

In [ ]:
# Sub-Step 6.1
# Tujuan: Grad-CAM (Captum) + fungsi centroid -- offset dari pusat gambar, satuan sama dengan center_offset EDA

from captum.attr import LayerGradCam, LayerAttribution, IntegratedGradients, Occlusion


def gradcam_heatmaps(model, layer, images_tensor, target_classes):
    """images_tensor: (N,3,H,W) sudah di-normalize. Return list of (H,W) numpy heatmap."""
    lgc = LayerGradCam(model, layer)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = lgc.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]))
        a = LayerAttribution.interpolate(a, (IMG_SIZE, IMG_SIZE))
        heatmaps.append(a.squeeze().detach().cpu().numpy())
    return heatmaps


def ig_heatmaps(model, images_tensor, target_classes, n_steps=20):
    ig = IntegratedGradients(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = ig.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]), n_steps=n_steps)
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


def occlusion_heatmaps(model, images_tensor, target_classes, window=32, stride=16):
    occ = Occlusion(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = occ.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]),
                           sliding_window_shapes=(3, window, window), strides=(3, stride, stride))
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


def heatmap_centroid_offset(heatmap):
    """Sama persis rumus center_offset di handcrafted_features -- supaya bisa dikorelasikan apple-to-apple."""
    h, w = heatmap.shape
    hm = np.clip(heatmap, 0, None)
    if hm.sum() <= 1e-8:
        return np.nan
    ys, xs = np.indices((h, w))
    cy = (ys * hm).sum() / hm.sum()
    cx = (xs * hm).sum() / hm.sum()
    return float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))

In [ ]:
# Sub-Step 6.2
# Tujuan: TCAV manual: layer activation + CAV (logistic regression) + directional derivative

from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_ind


GRAD_BATCH_SIZE = 32  # full run bisa proses ratusan gambar sekaligus -- batch kecil supaya
                       # tidak OOM saat forward+backward (backward jauh lebih boros memori
                       # daripada forward-only), lalu digabung.


def layer_activation_batch(model, layer, images_tensor):
    """Global-average-pooled activation di `layer`. Tanpa grad -- diproses per-batch kecil
    supaya aman untuk sample besar (full run), hasil digabung."""
    acts = {}
    def hook(m, i, o): acts["v"] = o.detach()
    h = layer.register_forward_hook(hook)
    outs = []
    with torch.no_grad():
        for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
            model(images_tensor[i:i + GRAD_BATCH_SIZE].to(device))
            a = acts["v"]
            a = a.mean(dim=[2, 3]) if a.dim() == 4 else (a.mean(dim=1) if a.dim() == 3 else a)
            outs.append(a.cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def layer_grad_wrt_target(model, layer, images_tensor, target_class):
    """Gradien logit target-class terhadap (GAP) activation layer -- dipakai sensitivitas TCAV.
    Diproses per-batch kecil (bukan semua sekaligus) karena backward pass jauh lebih boros
    memori GPU daripada forward-only -- ini yang menyebabkan OOM di full run sebelum diperbaiki."""
    acts = {}
    def hook(m, i, o):
        o.retain_grad()
        acts["v"] = o
    h = layer.register_forward_hook(hook)
    outs = []
    for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
        batch = images_tensor[i:i + GRAD_BATCH_SIZE].to(device)
        out = model(batch)
        logit = out[:, target_class].sum()
        model.zero_grad(set_to_none=True)
        logit.backward()
        a = acts["v"]
        grad = a.grad
        grad = grad.mean(dim=[2, 3]) if grad.dim() == 4 else (grad.mean(dim=1) if grad.dim() == 3 else grad)
        outs.append(grad.detach().cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def compute_cav(pos_acts, neg_acts, seed=SEED):
    X = np.concatenate([pos_acts, neg_acts], axis=0)
    y = np.concatenate([np.ones(len(pos_acts)), np.zeros(len(neg_acts))])
    clf = LogisticRegression(max_iter=1000, random_state=seed).fit(X, y)
    cav = clf.coef_[0]
    return cav / (np.linalg.norm(cav) + 1e-8)


def tcav_score_with_significance(model, layer, concept_pos_imgs, concept_neg_imgs,
                                  target_imgs, target_class, n_random=5):
    """Skor TCAV = fraksi gambar target yang sensitivitasnya positif terhadap arah konsep.
    Signifikansi: bandingkan ke n_random CAV acak (label pos/neg diacak dari pool yang sama)."""
    pos_acts = layer_activation_batch(model, layer, concept_pos_imgs)
    neg_acts = layer_activation_batch(model, layer, concept_neg_imgs)
    cav = compute_cav(pos_acts, neg_acts)
    grads = layer_grad_wrt_target(model, layer, target_imgs, target_class)
    sensitivities = grads @ cav
    real_score = float((sensitivities > 0).mean())

    pool_acts = np.concatenate([pos_acts, neg_acts], axis=0)
    n_pos = len(pos_acts)
    rng = np.random.RandomState(SEED)
    random_scores = []
    for _ in range(n_random):
        perm = rng.permutation(len(pool_acts))
        rand_cav = compute_cav(pool_acts[perm[:n_pos]], pool_acts[perm[n_pos:]])
        rand_sens = grads @ rand_cav
        random_scores.append(float((rand_sens > 0).mean()))
    _, p_value = ttest_ind([real_score], random_scores) if len(set(random_scores)) > 1 else (np.nan, np.nan)
    return {"tcav_score": real_score, "random_scores_mean": float(np.mean(random_scores)),
            "random_scores": random_scores, "p_value": float(p_value) if p_value == p_value else None}

In [ ]:
# Sub-Step 6.3
# Tujuan: Probe kausal: reposisi bean dalam frame & occlusion region

def probe_reposition(orig_path, shift_frac=0.25, margin=0.2):
    """Crop ulang dari gambar mentah di posisi asli vs digeser shift_frac*lebar_frame."""
    _, bbox = handcrafted_features(orig_path)
    img = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    x0, y0, x1, y1 = bbox
    bw, bh = max(x1 - x0, 1), max(y1 - y0, 1)
    half = max(bw, bh) * (1 + margin) / 2

    def crop_at(cx, cy):
        xa, ya = int(max(cx - half, 0)), int(max(cy - half, 0))
        xb, yb = int(min(cx + half, w)), int(min(cy + half, h))
        if xb <= xa or yb <= ya:
            return Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE))
        return Image.fromarray(img[ya:yb, xa:xb]).resize((IMG_SIZE, IMG_SIZE))

    cx0, cy0 = (x0 + x1) / 2, (y0 + y1) / 2
    return crop_at(cx0, cy0), crop_at(cx0 + shift_frac * w, cy0)


def probe_occlude(pil_img, region="center"):
    arr = np.array(pil_img).copy()
    h, w = arr.shape[:2]
    fill = arr.reshape(-1, arr.shape[-1]).mean(axis=0).astype(arr.dtype)
    slices = {
        "center": (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4)),
        "left": (slice(0, h), slice(0, w // 2)),
        "right": (slice(0, h), slice(w // 2, w)),
    }[region]
    arr[slices] = fill
    return Image.fromarray(arr)


def probe_color_jitter(pil_img, brightness_factor=1.3, saturation_factor=1.3):
    """Perturbasi warna DI LUAR rentang augmentasi training (training cuma brightness
    +-10%, saturation +-5% -- lihat Section 3). Hipotesis #1: warna adalah fitur EDA
    dengan effect size TERBESAR (eta^2=0.22-0.24) -- menguji apakah model rapuh terhadap
    perubahan warna yang lebih besar dari yang pernah dilihat saat training, indikasi
    ketergantungan berlebih pada warna sebagai sinyal (yang bisa jadi artefak sesi
    pemotretan, bukan cuma ciri bean asli)."""
    from PIL import ImageEnhance
    img = ImageEnhance.Brightness(pil_img).enhance(brightness_factor)
    img = ImageEnhance.Color(img).enhance(saturation_factor)
    return img


def prob_delta(predict_fn, pil_a, pil_b, class_idx):
    """|P(class_idx | a) - P(class_idx | b)| -- dipakai untuk semua probe di semua keluarga."""
    pa = predict_fn(pil_a)[class_idx]
    pb = predict_fn(pil_b)[class_idx]
    return float(abs(pa - pb)), float(pa), float(pb)

In [ ]:
# Sub-Step 6.4
# Tujuan: Embedding lineage: ekstraksi + nearest-neighbor label agreement

from sklearn.neighbors import NearestNeighbors


def extract_embeddings_for_df(model, layer, df, root_dir, transform, batch_size=32):
    loader = DataLoader(BeanDataset(df, root_dir, transform), batch_size=batch_size, shuffle=False, num_workers=2)
    embs = []
    for images, _, _ in loader:
        embs.append(layer_activation_batch(model, layer, images))
    return np.concatenate(embs, axis=0)


def nn_label_agreement(train_embs, train_labels, query_embs, query_labels, k=5):
    nn = NearestNeighbors(n_neighbors=k).fit(train_embs)
    _, idx = nn.kneighbors(query_embs)
    train_labels = np.asarray(train_labels)
    agree = np.array([(train_labels[row] == query_labels[i]).mean() for i, row in enumerate(idx)])
    return agree, idx

In [ ]:
# Sub-Step 6.5
# Tujuan: Helper agregat: korelasi Pearson terhadap metadata EDA, dengan penanganan NaN

from scipy.stats import pearsonr


def correlate_safe(values, meta_values):
    values = np.asarray(values, dtype=float)
    meta_values = np.asarray(meta_values, dtype=float)
    valid = ~(np.isnan(values) | np.isnan(meta_values))
    if valid.sum() < 3 or np.std(values[valid]) < 1e-8 or np.std(meta_values[valid]) < 1e-8:
        return np.nan, np.nan
    r, p = pearsonr(values[valid], meta_values[valid])
    return float(r), float(p)


def sample_per_class(df, n_per_class, seed=SEED):
    parts = []
    for cls in CLASS_NAMES:
        sub_df = df[df["label"] == cls]
        n = min(n_per_class, len(sub_df))
        parts.append(sub_df.sample(n=n, random_state=seed))
    return pd.concat(parts).reset_index(drop=True)

In [ ]:
# Sub-Step 6.6
# Tujuan: Concept split untuk TCAV -- dihitung dari gambar dataset_preprocessed (yang BENAR-BENAR dilihat model), bukan gambar mentah

# Catatan metodologis penting: crop-to-bbox di preprocess_dataset.py merekapkan bean ke tengah
# frame -- artinya center_offset di gambar RAW (orig_path) hampir tidak relevan lagi untuk
# model, karena model dilatih di dataset_preprocessed/ yang sudah direcenter. Concept split di
# bawah dihitung ULANG dari gambar dataset_preprocessed itu sendiri, supaya menguji variasi
# yang BENAR-BENAR bisa dilihat & dieksploitasi model, bukan variasi yang sudah hilang lewat
# preprocessing.

def compute_features_for_paths(paths):
    return pd.DataFrame([handcrafted_features(p)[0] for p in paths])


test_paths_prep = [PREP_DIR / p for p in test_df["image_path"]]
test_feats_prep = compute_features_for_paths(test_paths_prep)
test_feats_prep["image_path"] = test_df["image_path"].values
test_feats_prep["label"] = test_df["label"].values
print("Fitur (di gambar preprocessed) dihitung untuk", len(test_feats_prep), "gambar test.")
print(test_feats_prep[["center_offset", "bbox_ratio"]].describe().round(4))


def build_concept_examples(feature_col, concept_is_low, quantile=0.3, n=20):
    """concept_is_low=True -> positif = nilai fitur TERENDAH (mis. paling terpusat)."""
    vals = test_feats_prep[feature_col].values
    valid = ~np.isnan(vals)
    sub = test_feats_prep[valid].copy()
    lo_thr, hi_thr = sub[feature_col].quantile(quantile), sub[feature_col].quantile(1 - quantile)
    pos_df = sub[sub[feature_col] <= lo_thr] if concept_is_low else sub[sub[feature_col] >= hi_thr]
    neg_df = sub[sub[feature_col] >= hi_thr] if concept_is_low else sub[sub[feature_col] <= lo_thr]
    pos_df = pos_df.sample(n=min(n, len(pos_df)), random_state=SEED)
    neg_df = neg_df.sample(n=min(n, len(neg_df)), random_state=SEED)
    return pos_df["image_path"].tolist(), neg_df["image_path"].tolist()


CONCEPT_DEFS = [
    # Hipotesis #1 (kecurigaan awal): posisi bean sebagai shortcut framing (EDA v2 SS06)
    {"name": "off_center", "feature": "center_offset", "concept_is_low": True, "target_class": "premium"},
    {"name": "elongated_shape", "feature": "bbox_ratio", "concept_is_low": False, "target_class": "longberry"},
    # Hipotesis #4 (baru): warna sebagai artefak sesi pemotretan, bukan ciri bean asli --
    # mean_r/g/b adalah fitur dengan effect size TERBESAR di EDA (eta^2=0.22-0.24), belum
    # pernah diuji lewat TCAV sama sekali sebelumnya. "dark_color" diuji terhadap defect
    # karena warna gelap/kusam secara domain diasosiasikan dengan kerusakan.
    {"name": "dark_color", "feature": "mean_r", "concept_is_low": True, "target_class": "defect"},
    # Hipotesis #5 (baru): ukuran-di-frame sebagai shortcut terpisah dari posisi/bentuk --
    # peaberry (bulat) diuji karena bentuk bulat mengisi bounding box persegi lebih efisien
    # (rasio luas lingkaran/persegi ~0.785) dibanding bentuk elongated, independen dari
    # ukuran fisik bean yang sebenarnya.
    {"name": "large_area", "feature": "area_frac", "concept_is_low": False, "target_class": "peaberry"},
]
for cdef in CONCEPT_DEFS:
    pos_paths, neg_paths = build_concept_examples(cdef["feature"], cdef["concept_is_low"])
    cdef["pos_paths"] = pos_paths
    cdef["neg_paths"] = neg_paths
    print(f"Konsep '{cdef['name']}': {len(pos_paths)} positif, {len(neg_paths)} negatif")

In [ ]:
# Sub-Step 6.7
# Tujuan: Pipeline XAI generik untuk model neural (dipakai keluarga A, B, D per sub-model/head)

def load_pil_batch(paths, root_dir, transform):
    imgs = [transform(Image.open(root_dir / p if not str(p).startswith(str(root_dir)) else p).convert("RGB"))
            for p in paths]
    return torch.stack(imgs)


def xai_pipeline_neural(model, model_name, class_names, predict_fn, gradcam_layer=None,
                         embedding_layer=None, sample_df=None, lineage_train_df=None,
                         concept_defs=None, run_qualitative=True):
    """Satu pipeline untuk sudut pandang 1 (spasial), 2 (konsep), 3 (kausal), 4 (data lineage),
    diagregasi di sudut pandang 5. gradcam_layer=None -> lewati Grad-CAM (keluarga B/transformer).
    predict_fn(pil_image) -> vektor probabilitas 4-kelas, dipakai probe kausal."""
    result = {"model": model_name}
    sample_df = sample_df if sample_df is not None else sample_per_class(test_df, N_AGG_PER_CLASS)
    sample_paths = [PREP_DIR / p for p in sample_df["image_path"]]
    sample_labels = np.array([LABEL_TO_IDX[l] for l in sample_df["label"]])
    images_tensor = load_pil_batch(sample_df["image_path"].tolist(), PREP_DIR, eval_transform)

    # --- 1. Spasial: Grad-CAM (kalau ada conv layer) + centroid, dikorelasikan ke center_offset ---
    if gradcam_layer is not None:
        heatmaps = gradcam_heatmaps(model, gradcam_layer, images_tensor, sample_labels)
        centroids = np.array([heatmap_centroid_offset(hm) for hm in heatmaps])
        merged = sample_df.merge(test_feats_prep[["image_path", "center_offset"]], on="image_path", how="left")
        r, p = correlate_safe(centroids, merged["center_offset"].values)
        result["spatial_method"] = "GradCAM"
        result["centroid_offset_mean"] = float(np.nanmean(centroids))
        result["centroid_vs_center_offset_r"] = r
        result["centroid_vs_center_offset_p"] = p
    else:
        result["spatial_method"] = "IG+Occlusion (bukan Grad-CAM -- tidak ada conv feature map)"

    # --- 2. Konsep semantik: TCAV ---
    if concept_defs is not None and embedding_layer is not None:
        tcav_out = {}
        for cdef in concept_defs:
            pos_imgs = load_pil_batch(cdef["pos_paths"], PREP_DIR, eval_transform)
            neg_imgs = load_pil_batch(cdef["neg_paths"], PREP_DIR, eval_transform)
            target_idx = LABEL_TO_IDX[cdef["target_class"]]
            target_mask = sample_labels == target_idx
            target_imgs = images_tensor[target_mask] if target_mask.sum() >= 2 else images_tensor
            tcav_out[cdef["name"]] = tcav_score_with_significance(
                model, embedding_layer, pos_imgs, neg_imgs, target_imgs, target_idx
            )
        result["tcav"] = tcav_out

    # --- 3. Kausal: probe reposisi, occlusion, & warna -- diagregasi jadi rata-rata delta probabilitas ---
    reposition_deltas, occlusion_deltas, color_deltas = [], [], []
    for _, row in sample_df.iterrows():
        cls_idx = LABEL_TO_IDX[row["label"]]
        orig_raw_path = RAW_DIR / row["orig_path"]
        try:
            crop_a, crop_b = probe_reposition(orig_raw_path)
            d, _, _ = prob_delta(predict_fn, crop_a, crop_b, cls_idx)
            reposition_deltas.append(d)
        except Exception:
            pass
        prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        occluded = probe_occlude(prep_img, region="center")
        d2, _, _ = prob_delta(predict_fn, prep_img, occluded, cls_idx)
        occlusion_deltas.append(d2)
        jittered = probe_color_jitter(prep_img)
        d3, _, _ = prob_delta(predict_fn, prep_img, jittered, cls_idx)
        color_deltas.append(d3)
    result["probe_reposition_mean_delta"] = float(np.mean(reposition_deltas)) if reposition_deltas else np.nan
    result["probe_occlusion_center_mean_delta"] = float(np.mean(occlusion_deltas)) if occlusion_deltas else np.nan
    result["probe_color_jitter_mean_delta"] = float(np.mean(color_deltas)) if color_deltas else np.nan

    # --- 4. Data lineage: NN embedding label agreement ---
    if embedding_layer is not None and lineage_train_df is not None:
        train_embs = extract_embeddings_for_df(model, embedding_layer, lineage_train_df, PREP_DIR, eval_transform)
        train_labels_arr = np.array([LABEL_TO_IDX[l] for l in lineage_train_df["label"]])
        query_embs = extract_embeddings_for_df(model, embedding_layer, sample_df, PREP_DIR, eval_transform)
        agree, _ = nn_label_agreement(train_embs, train_labels_arr, query_embs, sample_labels, k=5)
        result["nn_lineage_label_agreement_mean"] = float(np.mean(agree))

    # --- Kualitatif: IG + Occlusion di N_QUAL contoh, untuk pemeriksaan visual manual ---
    if run_qualitative:
        qual_df = sample_per_class(test_df, max(1, N_QUAL // 4))
        qual_paths = qual_df["image_path"].tolist()
        qual_imgs = load_pil_batch(qual_paths, PREP_DIR, eval_transform)
        qual_labels = np.array([LABEL_TO_IDX[l] for l in qual_df["label"]])
        ig_maps = ig_heatmaps(model, qual_imgs, qual_labels)
        occ_maps = occlusion_heatmaps(model, qual_imgs, qual_labels)
        result["qualitative_ig_centroid_mean"] = float(np.nanmean([heatmap_centroid_offset(h) for h in ig_maps]))
        result["qualitative_occlusion_centroid_mean"] = float(np.nanmean([heatmap_centroid_offset(h) for h in occ_maps]))

    torch.cuda.empty_cache()  # cegah akumulasi cache antar model dalam loop keluarga A/B/D
    return result

In [ ]:
# Sub-Step 6.8
# Tujuan: predict_proba_pil(): wrapper generik model neural single-output -> vektor probabilitas, dipakai probe kausal

def predict_proba_pil(model, pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        return torch.softmax(model(x), dim=1).cpu().numpy()[0]


xai_results = {}  # diisi tiap keluarga -- dikonsolidasikan di Section 14

In [ ]:
# Sub-Step 6.9
# Tujuan: Helper overlay heatmap (dipakai visualisasi real_world di Section 13 & laporan di Section 15)

import matplotlib
matplotlib.use("Agg")
import matplotlib.cm as cm

FIG_DIR = Path("results/xai_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_VIZ_PER_CLASS = 1


def heatmap_overlay_rgb(pil_img, heatmap, alpha=0.5):
    img_arr = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")) / 255.0
    hm = heatmap - np.nanmin(heatmap)
    hm = hm / (np.nanmax(hm) + 1e-8)
    heat_colored = cm.jet(hm)[:, :, :3]
    overlay = (1 - alpha) * img_arr + alpha * heat_colored
    return np.clip(overlay, 0, 1)

## Section 7 — Keluarga A: CNN Polos (02, 03, 04, 05, 09)

Grad-CAM native (last conv layer), TCAV di layer yang sama, probe kausal, dan NN-embedding
lineage terhadap `fit_df` (697 gambar) sebagai referensi. Lihat `docs/xai-strategy.md` §4A.

In [ ]:
# Sub-Step 7.1
# Tujuan: Jalankan pipeline XAI generik untuk 02, 03, 04, 05, 09

FAMILY_A_LAYER = {
    "02_mobilenet_v3_large": lambda m: m.features[-1],
    "03_efficientnet_b0": lambda m: m.features[-1],
    "04_resnet18": lambda m: m.layer4[-1],
    "05_convnext_tiny": lambda m: m.features[-1],
    "09_noise_robust": lambda m: m.features[-1],
}

for name, layer_fn in FAMILY_A_LAYER.items():
    model = cnn_models[name]
    layer = layer_fn(model)
    predict_fn = (lambda m: (lambda img: predict_proba_pil(m, img)))(model)
    res = xai_pipeline_neural(
        model, name, CLASS_NAMES, predict_fn,
        gradcam_layer=layer, embedding_layer=layer,
        lineage_train_df=fit_df, concept_defs=CONCEPT_DEFS,
    )
    xai_results[name] = res
    print(f"--- {name} ---")
    for k, v in res.items():
        if k != "model" and not isinstance(v, dict):
            print(f"  {k}: {v}")
    if "tcav" in res:
        for cname, cval in res["tcav"].items():
            print(f"  tcav[{cname}]: score={cval['tcav_score']:.3f} vs random_mean={cval['random_scores_mean']:.3f} (p={cval['p_value']})")

## Section 8 — Keluarga B: Transformer (06 DeiT-Tiny)

**Bukan Grad-CAM** -- DeiT tidak punya feature map spasial konvolusi, jadi sudut pandang
spasial di sini hanya lewat Integrated Gradients & Occlusion (keduanya sudah dihitung sebagai
bagian "kualitatif" di `xai_pipeline_neural`, karena bekerja langsung di ruang piksel input,
model-agnostic). TCAV, probe kausal, dan lineage tetap jalan seperti biasa lewat representasi
token dari block transformer terakhir. Lihat `docs/xai-strategy.md` §4B.

In [ ]:
# Sub-Step 8.1
# Tujuan: Jalankan pipeline XAI (tanpa Grad-CAM) untuk 06_deit_tiny

deit_model = cnn_models["06_deit_tiny"]
deit_layer = deit_model.blocks[-1]  # output (B, n_token, C) -- layer_activation_batch rata-ratakan token
predict_fn_deit = lambda img: predict_proba_pil(deit_model, img)

res_deit = xai_pipeline_neural(
    deit_model, "06_deit_tiny", CLASS_NAMES, predict_fn_deit,
    gradcam_layer=None, embedding_layer=deit_layer,
    lineage_train_df=fit_df, concept_defs=CONCEPT_DEFS,
)
xai_results["06_deit_tiny"] = res_deit
print("--- 06_deit_tiny ---")
for k, v in res_deit.items():
    if k != "model" and not isinstance(v, dict):
        print(f"  {k}: {v}")
if "tcav" in res_deit:
    for cname, cval in res_deit["tcav"].items():
        print(f"  tcav[{cname}]: score={cval['tcav_score']:.3f} vs random_mean={cval['random_scores_mean']:.3f} (p={cval['p_value']})")

## Section 9 — Keluarga C: Tabular/Tree (01 Gradient Boosting)

Model ini tidak pernah melihat piksel -- input-nya 11 fitur hand-crafted. Spasial diganti
SHAP (analog paling setara: "apa yang mendorong keputusan", satuannya fitur bukan piksel).
Konsep (TCAV) dilewati -- fiturnya sendiri sudah konsep yang bisa dipahami manusia, memaksakan
TCAV di sini cuma menambah langkah tanpa informasi baru. Lihat `docs/xai-strategy.md` §4C.

In [ ]:
# Sub-Step 9.1
# Tujuan: SHAP TreeExplainer -- kontribusi 11 fitur, diagregasi per kelas

import shap

explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test)  # list per kelas ATAU array (n, features, n_class) tergantung versi shap
shap_values = np.array(shap_values)
if shap_values.ndim == 3 and shap_values.shape[0] == len(X_test):
    shap_values = np.transpose(shap_values, (2, 0, 1))  # -> (n_class, n_sample, n_feature)

mean_abs_shap_per_class = {}
for ci, cname in enumerate(CLASS_NAMES):
    class_mask = y_test == ci
    if class_mask.sum() == 0:
        continue
    vals = shap_values[ci][class_mask] if shap_values.ndim == 3 else shap_values[class_mask]
    mean_abs = np.abs(vals).mean(axis=0)
    mean_abs_shap_per_class[cname] = dict(zip(FEATURE_COLS, mean_abs.tolist()))

for cname, feat_importance in mean_abs_shap_per_class.items():
    top3 = sorted(feat_importance.items(), key=lambda kv: -kv[1])[:3]
    print(f"[{cname}] fitur paling penting (mean |SHAP|): {top3}")

In [ ]:
# Sub-Step 9.2
# Tujuan: Agregat: stabilitas peringkat SHAP -- apakah fitur teratas konsisten antar sampel satu kelas

def top_feature_stability(shap_vals_for_class):
    """Untuk tiap sampel, fitur mana yang |SHAP|-nya terbesar -- lalu hitung seberapa sering
    fitur yang SAMA muncul sebagai top-1 di seluruh sampel kelas itu (1.0 = selalu fitur yang
    sama, mendekati 1/11 = acak sepenuhnya)."""
    top1_idx = np.argmax(np.abs(shap_vals_for_class), axis=1)
    values, counts = np.unique(top1_idx, return_counts=True)
    dominant_frac = counts.max() / len(top1_idx)
    dominant_feature = FEATURE_COLS[values[counts.argmax()]]
    return dominant_feature, float(dominant_frac)


gb_stability = {}
for ci, cname in enumerate(CLASS_NAMES):
    class_mask = y_test == ci
    vals = shap_values[ci][class_mask] if shap_values.ndim == 3 else shap_values[class_mask]
    if len(vals) == 0:
        continue
    dom_feat, dom_frac = top_feature_stability(vals)
    gb_stability[cname] = {"dominant_feature": dom_feat, "fraction_of_samples": dom_frac}
    print(f"[{cname}] fitur top-1 paling sering: '{dom_feat}' ({dom_frac:.0%} sampel)")

In [ ]:
# Sub-Step 9.3
# Tujuan: Probe kausal: reposisi/occlusion gambar -> hitung ulang 11 fitur -> cek prediksi GB berubah

gb_reposition_deltas, gb_occlusion_deltas, gb_color_deltas = [], [], []
gb_sample_df = sample_per_class(test_df, N_AGG_PER_CLASS)
for _, row in gb_sample_df.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    orig_raw_path = RAW_DIR / row["orig_path"]

    def gb_predict_pil(pil_img):
        arr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        tmp_path = "/tmp/_probe_tmp.png"
        cv2.imwrite(tmp_path, arr)
        feats, _ = handcrafted_features(tmp_path)
        x = pd.DataFrame([feats])[FEATURE_COLS].values
        return gb_model.predict_proba(x)[0]

    try:
        crop_a, crop_b = probe_reposition(orig_raw_path)
        d, _, _ = prob_delta(gb_predict_pil, crop_a, crop_b, cls_idx)
        gb_reposition_deltas.append(d)
    except Exception:
        pass
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d2, _, _ = prob_delta(gb_predict_pil, prep_img, occluded, cls_idx)
    gb_occlusion_deltas.append(d2)
    jittered = probe_color_jitter(prep_img)
    d3, _, _ = prob_delta(gb_predict_pil, prep_img, jittered, cls_idx)
    gb_color_deltas.append(d3)

print(f"01_gradient_boosting probe_reposition_mean_delta: {np.mean(gb_reposition_deltas):.4f}")
print(f"01_gradient_boosting probe_occlusion_center_mean_delta: {np.mean(gb_occlusion_deltas):.4f}")
print(f"01_gradient_boosting probe_color_jitter_mean_delta: {np.mean(gb_color_deltas):.4f}")

In [ ]:
# Sub-Step 9.4
# Tujuan: Data lineage: vektor 11-fitur sebagai embedding, NN label agreement

gb_query_idx = gb_sample_df.index
X_query = build_feature_matrix(gb_sample_df)[0]
y_query = np.array([LABEL_TO_IDX[l] for l in gb_sample_df["label"]])
gb_nn_agreement, _ = nn_label_agreement(X_fit, y_fit, X_query, y_query, k=5)

xai_results["01_gradient_boosting"] = {
    "model": "01_gradient_boosting",
    "spatial_method": "SHAP (bukan Grad-CAM -- input berupa fitur tangan, bukan piksel)",
    "shap_top_features_per_class": mean_abs_shap_per_class,
    "shap_stability_per_class": gb_stability,
    "probe_reposition_mean_delta": float(np.mean(gb_reposition_deltas)) if gb_reposition_deltas else np.nan,
    "probe_occlusion_center_mean_delta": float(np.mean(gb_occlusion_deltas)),
    "probe_color_jitter_mean_delta": float(np.mean(gb_color_deltas)),
    "nn_lineage_label_agreement_mean": float(np.mean(gb_nn_agreement)),
}
print("01_gradient_boosting nn_lineage_label_agreement_mean:", xai_results["01_gradient_boosting"]["nn_lineage_label_agreement_mean"])

In [ ]:
# Sub-Step 9.5
# Tujuan: Padanan Hipotesis #1/#4/#5 untuk GB -- peringkat SHAP fitur yang bersangkutan (bukan dilewati, tapi dijawab langsung)

# GB tidak punya "layer" untuk CAV neural -- tapi karena inputnya cuma 11 fitur yang SUDAH
# jadi konsep yang bisa dipahami manusia (bukan representasi laten), peringkat kontribusi
# SHAP untuk fitur yang bersangkutan ADALAH jawaban langsungnya untuk tiap hipotesis,
# bukan pendekatan yang lebih lemah dari TCAV -- lihat docs/xai-strategy.md SS4C.
# Kunci dict ini sengaja disamakan persis dengan cdef["name"] di CONCEPT_DEFS (Section 6.6)
# supaya konsisten dengan kolom tcav_<nama> di keluarga lain -- bukan cuma label tampilan.
CONCEPT_TO_FEATURE = {
    "off_center": "center_offset", "elongated_shape": "bbox_ratio",
    "dark_color": "mean_r", "large_area": "area_frac",
}
HYPOTHESIS_LABEL = {"off_center": "H1", "dark_color": "H4", "large_area": "H5"}
gb_concept_rank = {}
for concept_name, feat in CONCEPT_TO_FEATURE.items():
    per_class_rank = {}
    for cname, importance in mean_abs_shap_per_class.items():
        ranked = sorted(importance.items(), key=lambda kv: -kv[1])
        rank = [f for f, _ in ranked].index(feat) + 1
        per_class_rank[cname] = rank
    gb_concept_rank[concept_name] = per_class_rank
    hyp_tag = f" ({HYPOTHESIS_LABEL[concept_name]})" if concept_name in HYPOTHESIS_LABEL else ""
    print(f"[{concept_name}{hyp_tag} -> {feat}] peringkat kepentingan (1=terpenting) per kelas: {per_class_rank}")

xai_results["01_gradient_boosting"]["concept_equivalent_shap_rank"] = gb_concept_rank

## Section 10 — Keluarga D: Multi-Submodel (07 Hierarchical, 08 Multi-Task)

Data lineage TIDAK diulang di sini -- kedua sub-model 07/08 memakai arsitektur EfficientNet-B0
yang SAMA seperti model 03/09 yang sudah dianalisis lengkap di Section 7, dan near-duplicate
cross-class adalah properti DATASET (bukan model), jadi mengulang lineage per sub-model tidak
menambah temuan baru. TCAV (hipotesis #1/#4/#5) SEKARANG tetap dijalankan di 10.5 per instruksi
cakupan penuh -- walau representasinya mirip 03/09, tiap sub-model punya output space sendiri
(mis. damage_model cuma 2 kelas) sehingga skornya tidak otomatis identik dan harus dibuktikan,
bukan diasumsikan. Fokus utama keluarga D tetap pertanyaan yang UNIK untuknya: **apakah kedua
head/sub-model benar-benar melihat wilayah gambar yang berbeda** (spesialisasi spasial sesuai
desain), lewat korelasi piksel-demi-piksel antar heatmap Grad-CAM keduanya.
Lihat `docs/xai-strategy.md` §4D.

In [ ]:
# Sub-Step 10.1
# Tujuan: Grad-CAM per sub-model/head + korelasi (head-agreement) -- Model 07 Hierarchical

sample_nondefect = sample_per_class(test_df[test_df["label"] != "defect"], N_AGG_PER_CLASS)
images_nd = load_pil_batch(sample_nondefect["image_path"].tolist(), PREP_DIR, eval_transform)
type_targets = np.array([TYPE_LABEL_FN(l) for l in sample_nondefect["label"]])
damage_targets = np.zeros(len(sample_nondefect), dtype=int)  # semua non-defect -> target kelas "intact" (0)

heatmaps_type = gradcam_heatmaps(type_model, type_model.features[-1], images_nd, type_targets)
heatmaps_damage = gradcam_heatmaps(damage_model, damage_model.features[-1], images_nd, damage_targets)

head_corrs_07 = []
for ha, hb in zip(heatmaps_type, heatmaps_damage):
    if np.std(ha) < 1e-8 or np.std(hb) < 1e-8:
        continue
    head_corrs_07.append(float(np.corrcoef(ha.flatten(), hb.flatten())[0, 1]))
head_agreement_07 = float(np.mean(head_corrs_07)) if head_corrs_07 else np.nan
print(f"07_hierarchical: korelasi heatmap head-tipe vs head-rusak = {head_agreement_07:.4f} "
      f"({'mirip -- kedua sub-model melihat area yang sama' if head_agreement_07 > 0.5 else 'berbeda -- spesialisasi spasial nyata'})")

In [ ]:
# Sub-Step 10.2
# Tujuan: Probe kausal Model 07 (gabungan proba damage x type)

def hierarchical_predict_proba_pil(pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        p_damage = torch.softmax(damage_model(x), dim=1)[0]
        p_type = torch.softmax(type_model(x), dim=1)[0]
    proba = np.zeros(4)
    proba[LABEL_TO_IDX["defect"]] = p_damage[1].item()
    not_defect = p_damage[0].item()
    proba[LABEL_TO_IDX["premium"]] = not_defect * p_type[0].item()
    proba[LABEL_TO_IDX["peaberry"]] = not_defect * p_type[1].item()
    proba[LABEL_TO_IDX["longberry"]] = not_defect * p_type[2].item()
    return proba


sample_07 = sample_per_class(test_df, N_AGG_PER_CLASS)
reposition_deltas_07, occlusion_deltas_07, color_deltas_07 = [], [], []
for _, row in sample_07.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    try:
        crop_a, crop_b = probe_reposition(RAW_DIR / row["orig_path"])
        d, _, _ = prob_delta(hierarchical_predict_proba_pil, crop_a, crop_b, cls_idx)
        reposition_deltas_07.append(d)
    except Exception:
        pass
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d2, _, _ = prob_delta(hierarchical_predict_proba_pil, prep_img, occluded, cls_idx)
    occlusion_deltas_07.append(d2)
    jittered = probe_color_jitter(prep_img)
    d3, _, _ = prob_delta(hierarchical_predict_proba_pil, prep_img, jittered, cls_idx)
    color_deltas_07.append(d3)

xai_results["07_hierarchical"] = {
    "model": "07_hierarchical",
    "spatial_method": "GradCAM per sub-model (tipe & rusak terpisah)",
    "head_agreement_heatmap_corr": head_agreement_07,
    "probe_reposition_mean_delta": float(np.mean(reposition_deltas_07)) if reposition_deltas_07 else np.nan,
    "probe_occlusion_center_mean_delta": float(np.mean(occlusion_deltas_07)),
    "probe_color_jitter_mean_delta": float(np.mean(color_deltas_07)),
}
print("07_hierarchical:", {k: v for k, v in xai_results["07_hierarchical"].items() if k != "model"})
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 10.3
# Tujuan: Grad-CAM per head (wrapper output tunggal) + head-agreement -- Model 08 Multi-task

class HeadWrapper(nn.Module):
    def __init__(self, base, head_idx):
        super().__init__()
        self.base = base
        self.head_idx = head_idx

    def forward(self, x):
        outs = self.base(x)
        return outs[self.head_idx]


damage_wrapper = HeadWrapper(mt_model, 0).to(device).eval()
type_wrapper = HeadWrapper(mt_model, 1).to(device).eval()
mt_layer = mt_model.backbone.features[-1]

heatmaps_damage_08 = gradcam_heatmaps(damage_wrapper, mt_layer, images_nd, damage_targets)
heatmaps_type_08 = gradcam_heatmaps(type_wrapper, mt_layer, images_nd, type_targets)

head_corrs_08 = []
for ha, hb in zip(heatmaps_damage_08, heatmaps_type_08):
    if np.std(ha) < 1e-8 or np.std(hb) < 1e-8:
        continue
    head_corrs_08.append(float(np.corrcoef(ha.flatten(), hb.flatten())[0, 1]))
head_agreement_08 = float(np.mean(head_corrs_08)) if head_corrs_08 else np.nan
print(f"08_multitask: korelasi heatmap head-rusak vs head-tipe (BERBAGI 1 backbone) = {head_agreement_08:.4f} "
      f"({'mirip -- wajar krn backbone dibagi' if head_agreement_08 > 0.5 else 'berbeda meski backbone sama'})")

In [ ]:
# Sub-Step 10.4
# Tujuan: Probe kausal Model 08 (gabungan proba damage x type dari 1 backbone)

def multitask_predict_proba_pil(pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out_damage, out_type = mt_model(x)
        p_damage = torch.softmax(out_damage, dim=1)[0]
        p_type = torch.softmax(out_type, dim=1)[0]
    proba = np.zeros(4)
    proba[LABEL_TO_IDX["defect"]] = p_damage[1].item()
    not_defect = p_damage[0].item()
    proba[LABEL_TO_IDX["premium"]] = not_defect * p_type[0].item()
    proba[LABEL_TO_IDX["peaberry"]] = not_defect * p_type[1].item()
    proba[LABEL_TO_IDX["longberry"]] = not_defect * p_type[2].item()
    return proba


reposition_deltas_08, occlusion_deltas_08, color_deltas_08 = [], [], []
for _, row in sample_07.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    try:
        crop_a, crop_b = probe_reposition(RAW_DIR / row["orig_path"])
        d, _, _ = prob_delta(multitask_predict_proba_pil, crop_a, crop_b, cls_idx)
        reposition_deltas_08.append(d)
    except Exception:
        pass
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d2, _, _ = prob_delta(multitask_predict_proba_pil, prep_img, occluded, cls_idx)
    occlusion_deltas_08.append(d2)
    jittered = probe_color_jitter(prep_img)
    d3, _, _ = prob_delta(multitask_predict_proba_pil, prep_img, jittered, cls_idx)
    color_deltas_08.append(d3)

xai_results["08_multitask"] = {
    "model": "08_multitask",
    "spatial_method": "GradCAM per head (wrapper, 1 backbone dibagi)",
    "head_agreement_heatmap_corr": head_agreement_08,
    "probe_reposition_mean_delta": float(np.mean(reposition_deltas_08)) if reposition_deltas_08 else np.nan,
    "probe_occlusion_center_mean_delta": float(np.mean(occlusion_deltas_08)),
    "probe_color_jitter_mean_delta": float(np.mean(color_deltas_08)),
}
print("08_multitask:", {k: v for k, v in xai_results["08_multitask"].items() if k != "model"})
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 10.5
# Tujuan: TCAV Hipotesis #1/#4/#5 untuk sub-model/head 07 & 08 -- hanya konsep yang target-kelasnya ADA di ruang output sub-model itu

# type_model/type_wrapper cuma 3-kelas (premium/peaberry/longberry) -- dark_color->defect
# TIDAK ADA di ruang kelasnya (N/A, bukan dilewati karena malas). damage_model/damage_wrapper
# cuma 2-kelas (intact/defect) -- off_center/elongated_shape/large_area TIDAK ADA di ruang
# kelasnya. Ini keterbatasan arsitektural nyata, bukan penyempitan cakupan demi kepraktisan.
SUBMODEL_TCAV_TARGETS = {
    "type": {"off_center": 0, "elongated_shape": 2, "large_area": 1},   # index lokal TYPE_MAP; KUNCI HARUS
    "damage": {"dark_color": 1},                                        # persis cdef["name"] di CONCEPT_DEFS
}


def run_tcav_on_submodel(model, layer, target_map, sample_labels_local, images_tensor):
    out = {}
    for cdef in CONCEPT_DEFS:
        local_idx = target_map.get(cdef["name"])
        if local_idx is None:
            out[cdef["name"]] = None  # N/A -- target-kelas tidak ada di ruang output sub-model ini
            continue
        pos_imgs = load_pil_batch(cdef["pos_paths"], PREP_DIR, eval_transform)
        neg_imgs = load_pil_batch(cdef["neg_paths"], PREP_DIR, eval_transform)
        target_mask = sample_labels_local == local_idx
        target_imgs = images_tensor[target_mask] if target_mask.sum() >= 2 else images_tensor
        out[cdef["name"]] = tcav_score_with_significance(model, layer, pos_imgs, neg_imgs, target_imgs, local_idx)
    return out


tcav_07_type = run_tcav_on_submodel(type_model, type_model.features[-1], SUBMODEL_TCAV_TARGETS["type"], type_targets, images_nd)
tcav_07_damage = run_tcav_on_submodel(damage_model, damage_model.features[-1], SUBMODEL_TCAV_TARGETS["damage"], damage_targets, images_nd)
tcav_08_type = run_tcav_on_submodel(type_wrapper, mt_layer, SUBMODEL_TCAV_TARGETS["type"], type_targets, images_nd)
tcav_08_damage = run_tcav_on_submodel(damage_wrapper, mt_layer, SUBMODEL_TCAV_TARGETS["damage"], damage_targets, images_nd)
torch.cuda.empty_cache()

xai_results["07_hierarchical"]["tcav_type_submodel"] = tcav_07_type
xai_results["07_hierarchical"]["tcav_damage_submodel"] = tcav_07_damage
xai_results["08_multitask"]["tcav_type_head"] = tcav_08_type
xai_results["08_multitask"]["tcav_damage_head"] = tcav_08_damage

for label, res in [("07 type", tcav_07_type), ("07 damage", tcav_07_damage), ("08 type", tcav_08_type), ("08 damage", tcav_08_damage)]:
    summary = {k: (f"score={v['tcav_score']:.3f} p={v['p_value']}" if v else "N/A") for k, v in res.items()}
    print(f"[{label}] {summary}")

## Section 11 — Keluarga E: Ensemble (10)

Bukan model tunggal -- gabungan Gradient Boosting (#01) + ResNet-18 (#04, CNN yang dipilih
saat ensemble asli dibentuk). Penjelasannya adalah gabungan SHAP (Section 9) + Grad-CAM
(Section 7) berdampingan. Yang baru di sini: **agregat kesepakatan prediksi** GB vs ResNet-18
di SELURUH test set (231 gambar) -- diagnostik langsung untuk temuan laporan training bahwa
ensemble (0,891) lebih jelek dari ResNet-18 sendirian (0,926). Lihat `docs/xai-strategy.md` §4E.

In [ ]:
# Sub-Step 11.1
# Tujuan: Agregat: tingkat setuju/tidak-setuju GB vs ResNet-18 di seluruh test set, disilangkan dengan ketepatan

gb_test_pred_full = gb_model.predict(X_test)

@torch.no_grad()
def predict_all_flat(model, loader):
    model.eval()
    preds = []
    for images, _, _ in loader:
        preds.append(model(images.to(device)).argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)

resnet_model = cnn_models["04_resnet18"]
resnet_test_pred_full = predict_all_flat(resnet_model, test_loader)

agree_mask = gb_test_pred_full == resnet_test_pred_full
resnet_correct = resnet_test_pred_full == y_test
gb_correct = gb_test_pred_full == y_test

resnet_correct_gb_disagree = resnet_correct & (~agree_mask)
resnet_correct_and_agree = resnet_correct & agree_mask

agreement_rate = float(agree_mask.mean())
resnet_correct_n = int(resnet_correct.sum())
disagree_when_resnet_correct = int(resnet_correct_gb_disagree.sum())

print(f"Tingkat kesepakatan GB vs ResNet-18 di seluruh test set: {agreement_rate:.2%}")
print(f"Dari {resnet_correct_n} kasus ResNet-18 benar: GB TIDAK sepakat di {disagree_when_resnet_correct} kasus "
      f"({disagree_when_resnet_correct/resnet_correct_n:.2%}) -- ini yang berpotensi menarik rata-rata ensemble ke arah salah.")

xai_results["10_ensemble"] = {
    "model": "10_ensemble",
    "spatial_method": "Gabungan SHAP (01) + GradCAM (04_resnet18) berdampingan",
    "gb_resnet_agreement_rate": agreement_rate,
    "resnet_correct_but_gb_disagrees_rate": float(disagree_when_resnet_correct / resnet_correct_n) if resnet_correct_n else np.nan,
}

## Section 12 — Hipotesis #2: Kebocoran Konsep "Kerusakan" ke Kelas Non-Defect

EDA v3 mencurigai `defect` sebenarnya campuran 3 jenis lain yang rusak (3 metode independen
sepakat). Kalau benar, model mungkin punya SATU arah representasi "rusak" yang seharusnya
cuma aktif untuk gambar `defect`, tapi ikut ter-aktivasi (bocor) pada gambar non-defect yang
kebetulan punya blemish/cacat visual minor. Ini beda dari Section 10 (spesialisasi SPASIAL 2
head Model 07/08) — di sini yang diuji adalah SATU konsep tunggal, dievaluasi lintas kelas.

**Diuji di SEMUA 10 model, tanpa kecuali**, dengan adapter sesuai keluarga arsitektur (rincian
di tiap sub-step): neural via CAV+gradien (A/B/D), Gradient Boosting via proyeksi linear di
ruang 11-fitur (padanan CAV untuk model non-differentiable), Ensemble via dampak nyata gabungan.
Satu pengecualian arsitektural murni: sub-model `type` (07a, 08 head-tipe) tidak punya kelas
`defect` sama sekali di ruang outputnya (3-kelas: premium/peaberry/longberry) — secara harfiah
tidak bisa "bocor ke defect" karena defect bukan salah satu pilihannya.

In [ ]:
# Sub-Step 12.1
# Tujuan: Fungsi generik damage-leak untuk model neural (dipakai keluarga A, B, dan sub-model D yang punya kelas defect)

def damage_leak_test(model, layer, model_name, defect_idx):
    """CAV dilatih dari contoh defect (positif) vs non-defect (negatif) -- arah 'rusak' di
    ruang representasi model. Diuji: seberapa besar gambar NON-DEFECT sensitif ke arah itu,
    dan apakah sensitivitas tinggi berkorelasi dengan gambar itu benar-benar SALAH
    diklasifikasikan jadi defect. `defect_idx` beda-beda per model (4-kelas flat vs 2-kelas
    biner untuk sub-model 07b/08-damage) -- makanya jadi parameter, bukan konstanta."""
    defect_paths = test_df[test_df["label"] == "defect"]["image_path"].tolist()
    nondefect_df = test_df[test_df["label"] != "defect"]
    nondefect_paths_all = nondefect_df["image_path"].tolist()

    rng = np.random.RandomState(SEED)
    n_concept = min(30, len(defect_paths), len(nondefect_paths_all))
    defect_sample = rng.choice(defect_paths, n_concept, replace=False)
    nondefect_sample_for_cav = rng.choice(nondefect_paths_all, n_concept, replace=False)

    defect_imgs = load_pil_batch(list(defect_sample), PREP_DIR, eval_transform)
    nondefect_imgs_cav = load_pil_batch(list(nondefect_sample_for_cav), PREP_DIR, eval_transform)
    damage_cav = compute_cav(
        layer_activation_batch(model, layer, defect_imgs),
        layer_activation_batch(model, layer, nondefect_imgs_cav),
    )

    eval_df = sample_per_class(nondefect_df, N_AGG_PER_CLASS)
    eval_imgs = load_pil_batch(eval_df["image_path"].tolist(), PREP_DIR, eval_transform)
    grads = layer_grad_wrt_target(model, layer, eval_imgs, defect_idx)
    sensitivities = grads @ damage_cav

    predict_fn = lambda img: predict_proba_pil(model, img)
    predicted_idx = np.array([
        predict_fn(Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))).argmax()
        for p in eval_df["image_path"]
    ])
    misclassified_as_defect = (predicted_idx == defect_idx).astype(float)

    leak_score = float((sensitivities > 0).mean())
    r, p = correlate_safe(sensitivities, misclassified_as_defect)
    n_misclassified = int(misclassified_as_defect.sum())
    print(f"[{model_name}] Damage-leak score: {leak_score:.3f} | {n_misclassified}/{len(eval_df)} non-defect justru diprediksi defect | r={r}, p={p}")
    return {"model": model_name, "damage_leak_score": leak_score,
            "n_misclassified_as_defect": n_misclassified, "n_eval": len(eval_df),
            "damage_leak_vs_misclass_r": r, "damage_leak_vs_misclass_p": p}


hypothesis2_results = []
for _name in ["02_mobilenet_v3_large", "03_efficientnet_b0", "04_resnet18", "05_convnext_tiny", "09_noise_robust"]:
    _model = cnn_models[_name]
    _layer = FAMILY_A_LAYER[_name](_model)
    hypothesis2_results.append(damage_leak_test(_model, _layer, _name, LABEL_TO_IDX["defect"]))
    torch.cuda.empty_cache()

hypothesis2_results.append(damage_leak_test(deit_model, deit_layer, "06_deit_tiny", LABEL_TO_IDX["defect"]))
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 12.2
# Tujuan: Family D: damage-leak pada sub-model/head yang punya kelas defect (07b_damage, 08 head-rusak) -- type/type-head N/A (tidak punya kelas defect)

hypothesis2_results.append(damage_leak_test(damage_model, damage_model.features[-1], "07b_damage", defect_idx=1))
hypothesis2_results.append(damage_leak_test(damage_wrapper, mt_layer, "08_damage_head", defect_idx=1))
torch.cuda.empty_cache()
print("07a_type & 08_type_head: N/A -- ruang output sub-model ini (premium/peaberry/longberry) tidak punya kelas 'defect' sama sekali.")

In [ ]:
# Sub-Step 12.3
# Tujuan: Gradient Boosting: padanan CAV via proyeksi linear di ruang 11-fitur (model non-differentiable, tidak punya gradien)

def damage_leak_test_gb():
    """GB tidak punya gradien -- padanannya: latih arah 'rusak' via compute_cav() di ruang
    11-fitur (fungsi yang SAMA dipakai model neural, cuma inputnya vektor fitur bukan
    aktivasi layer), lalu proyeksikan (dot product) sampel non-defect ke arah itu sebagai
    analog 'sensitivitas' -- tanpa gradien, tapi mekanismenya sejalan."""
    defect_mask_fit = y_fit == LABEL_TO_IDX["defect"]
    damage_cav_gb = compute_cav(X_fit[defect_mask_fit], X_fit[~defect_mask_fit])

    nondefect_test_mask = y_test != LABEL_TO_IDX["defect"]
    X_nondefect = X_test[nondefect_test_mask]
    sensitivities = X_nondefect @ damage_cav_gb

    predicted_idx = gb_model.predict(X_nondefect)
    misclassified_as_defect = (predicted_idx == LABEL_TO_IDX["defect"]).astype(float)

    leak_score = float((sensitivities > 0).mean())
    r, p = correlate_safe(sensitivities, misclassified_as_defect)
    n_misclassified = int(misclassified_as_defect.sum())
    print(f"[01_gradient_boosting] Damage-leak score (proyeksi fitur): {leak_score:.3f} | "
          f"{n_misclassified}/{len(X_nondefect)} non-defect justru diprediksi defect | r={r}, p={p}")
    return {"model": "01_gradient_boosting", "damage_leak_score": leak_score,
            "n_misclassified_as_defect": n_misclassified, "n_eval": int(len(X_nondefect)),
            "damage_leak_vs_misclass_r": r, "damage_leak_vs_misclass_p": p}


hypothesis2_results.append(damage_leak_test_gb())

In [ ]:
# Sub-Step 12.4
# Tujuan: Ensemble: dampak nyata gabungan (tidak ada representasi tunggal untuk dihitung sensitivitasnya)

def damage_leak_test_ensemble():
    """Ensemble bukan model tunggal -- tidak ada satu ruang representasi untuk CAV. Yang
    bisa dihitung: dampak NYATA gabungan (proba rata-rata GB+ResNet-18, bobot 0,5/0,5 --
    sama seperti definisi ensemble asli di CBQD - Training.ipynb) pada sampel non-defect
    yang sama, konsisten dengan Section 11 memperlakukan Ensemble sebagai gabungan C+A."""
    nondefect_test_mask = y_test != LABEL_TO_IDX["defect"]
    X_nondefect = X_test[nondefect_test_mask]
    nondefect_df_ens = test_df[nondefect_test_mask].reset_index(drop=True)

    gb_proba_nd = gb_model.predict_proba(X_nondefect)
    resnet_proba_nd = np.array([
        predict_proba_pil(resnet_model, Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        for p in nondefect_df_ens["image_path"]
    ])
    ensemble_proba_nd = 0.5 * gb_proba_nd + 0.5 * resnet_proba_nd
    predicted_idx = ensemble_proba_nd.argmax(axis=1)
    misclassified_as_defect = (predicted_idx == LABEL_TO_IDX["defect"]).astype(float)
    n_misclassified = int(misclassified_as_defect.sum())
    n_eval = int(nondefect_test_mask.sum())
    print(f"[10_ensemble] {n_misclassified}/{n_eval} non-defect justru diprediksi defect (tidak ada skor sensitivitas -- bukan model tunggal)")
    return {"model": "10_ensemble", "damage_leak_score": np.nan,
            "n_misclassified_as_defect": n_misclassified, "n_eval": n_eval,
            "damage_leak_vs_misclass_r": np.nan, "damage_leak_vs_misclass_p": np.nan}


hypothesis2_results.append(damage_leak_test_ensemble())

## Section 13 — Hipotesis #3: Generalisasi ke `real_world/` (Sanity Check)

Seluruh Section 6–12 dijalankan di `test/` — yang berasal dari distribusi TRAIN yang sama
(sama-sama hasil `dataset_preprocessed/`, dipisah lewat cluster-aware split). `real_world/`
(200 gambar, sengaja dipisahkan sejak awal karena kondisi pengambilan gambarnya berbeda,
TANPA label) belum pernah disentuh sama sekali oleh XAI manapun. "Lolos XAI di test" belum
tentu "lolos di kondisi dunia nyata" — section ini pemeriksaan pertama untuk celah itu.

In [ ]:
# Sub-Step 13.1
# Tujuan: Muat real_world (200 gambar, tanpa label); helper generik confidence & distribusi kelas

real_world_df = manifest[manifest["split"] == "real_world"].reset_index(drop=True)
print(f"real_world: {len(real_world_df)} gambar (tanpa label, tidak pernah dipakai training/evaluasi)")

real_world_loader = DataLoader(
    BeanDataset(real_world_df, PREP_DIR, eval_transform, label_fn=lambda l: 0),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
)  # label_fn dummy -- real_world tidak punya ground truth, prediksi labelnya TIDAK dipakai


@torch.no_grad()
def predict_proba_loader(model, loader):
    model.eval()
    all_proba = []
    for images, _, _ in loader:
        all_proba.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(all_proba, axis=0)


def summarize_hypothesis3(name, test_proba, rw_proba):
    """Dipakai SEMUA model -- test_proba/rw_proba selalu array (n, 4) probabilitas flat,
    apa pun mekanisme internal model penghasilnya (langsung softmax, gabungan sub-model,
    atau gabungan ensemble)."""
    test_conf, rw_conf = test_proba.max(axis=1), rw_proba.max(axis=1)
    rw_pred = rw_proba.argmax(axis=1)
    rw_class_dist = {cname: float((rw_pred == i).mean()) for i, cname in enumerate(CLASS_NAMES)}
    test_class_dist = {cname: float((y_test == i).mean()) for i, cname in enumerate(CLASS_NAMES)}
    print(f"--- {name} --- Confidence test: {test_conf.mean():.3f} | real_world: {rw_conf.mean():.3f}")
    print(f"  Distribusi prediksi real_world: {rw_class_dist}")
    return {"model": name, "test_confidence_mean": float(test_conf.mean()),
            "real_world_confidence_mean": float(rw_conf.mean()),
            "real_world_class_distribution": rw_class_dist, "test_class_distribution": test_class_dist}


hypothesis3_results = []

In [ ]:
# Sub-Step 13.2
# Tujuan: Keluarga A & B (02,03,04,05,06,09): forward pass generik, murah, tanpa kecuali

for _name in ["02_mobilenet_v3_large", "03_efficientnet_b0", "04_resnet18", "05_convnext_tiny", "09_noise_robust"]:
    _model = cnn_models[_name]
    rw_proba = predict_proba_loader(_model, real_world_loader)
    test_proba = predict_proba_loader(_model, test_loader)
    hypothesis3_results.append(summarize_hypothesis3(_name, test_proba, rw_proba))
    torch.cuda.empty_cache()

rw_proba_deit = predict_proba_loader(deit_model, real_world_loader)
test_proba_deit = predict_proba_loader(deit_model, test_loader)
hypothesis3_results.append(summarize_hypothesis3("06_deit_tiny", test_proba_deit, rw_proba_deit))
torch.cuda.empty_cache()

In [ ]:
# Sub-Step 13.3
# Tujuan: Gradient Boosting: hitung 11 fitur untuk real_world dari gambar mentah (orig_path tersedia untuk semua split)

X_realworld = pd.DataFrame(
    [handcrafted_features(RAW_DIR / p)[0] for p in real_world_df["orig_path"]]
)[FEATURE_COLS].values

gb_test_proba_full = gb_model.predict_proba(X_test)
gb_rw_proba = gb_model.predict_proba(X_realworld)
hypothesis3_results.append(summarize_hypothesis3("01_gradient_boosting", gb_test_proba_full, gb_rw_proba))

In [ ]:
# Sub-Step 13.4
# Tujuan: Family D (07 Hierarchical, 08 Multi-task): loop per-gambar memakai combiner yang sudah ada

def batch_predict_proba_pil_fn(predict_fn, df, root_dir):
    return np.array([
        predict_fn(Image.open(root_dir / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        for p in df["image_path"]
    ])


test_proba_07 = batch_predict_proba_pil_fn(hierarchical_predict_proba_pil, test_df, PREP_DIR)
rw_proba_07 = batch_predict_proba_pil_fn(hierarchical_predict_proba_pil, real_world_df, PREP_DIR)
hypothesis3_results.append(summarize_hypothesis3("07_hierarchical", test_proba_07, rw_proba_07))

test_proba_08 = batch_predict_proba_pil_fn(multitask_predict_proba_pil, test_df, PREP_DIR)
rw_proba_08 = batch_predict_proba_pil_fn(multitask_predict_proba_pil, real_world_df, PREP_DIR)
hypothesis3_results.append(summarize_hypothesis3("08_multitask", test_proba_08, rw_proba_08))

In [ ]:
# Sub-Step 13.5
# Tujuan: Ensemble: gabungan proba GB + ResNet-18 (0,5/0,5) di test DAN real_world

resnet_test_proba_full = predict_proba_loader(resnet_model, test_loader)
resnet_rw_proba = predict_proba_loader(resnet_model, real_world_loader)

ensemble_test_proba = 0.5 * gb_test_proba_full + 0.5 * resnet_test_proba_full
ensemble_rw_proba = 0.5 * gb_rw_proba + 0.5 * resnet_rw_proba
hypothesis3_results.append(summarize_hypothesis3("10_ensemble", ensemble_test_proba, ensemble_rw_proba))

In [ ]:
# Sub-Step 13.6
# Tujuan: Visual: Grad-CAM pada contoh real_world (09_noise_robust) -- apakah tetap fokus ke bean?

rw_viz_df = real_world_df.sample(n=min(8, len(real_world_df)), random_state=SEED).reset_index(drop=True)
rw_viz_imgs = load_pil_batch(rw_viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
rw_viz_pil = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in rw_viz_df["image_path"]]

_rw_model = cnn_models["09_noise_robust"]
_rw_layer = FAMILY_A_LAYER["09_noise_robust"](_rw_model)
rw_proba_viz = np.array([predict_proba_pil(_rw_model, img) for img in rw_viz_pil])
rw_pred_viz = rw_proba_viz.argmax(axis=1)
rw_heatmaps = gradcam_heatmaps(_rw_model, _rw_layer, rw_viz_imgs, rw_pred_viz)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for i in range(len(rw_viz_pil)):
    r, c = divmod(i, 4)
    axes[r, c].imshow(heatmap_overlay_rgb(rw_viz_pil[i], rw_heatmaps[i]))
    axes[r, c].set_title(f"pred={CLASS_NAMES[rw_pred_viz[i]]}\nP={rw_proba_viz[i][rw_pred_viz[i]]:.2f}", fontsize=9)
    axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
fig.suptitle("Grad-CAM pada real_world/ (09_noise_robust) -- sanity check generalisasi", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "real_world_gradcam.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'real_world_gradcam.png'}")

## Section 14 — Konsolidasi & Kesimpulan

Menggabungkan temuan dari kelima keluarga jadi satu tabel, disimpan ke
`metadata/xai_summary.csv` (disalin ke repo lokal & commit, sama seperti
`metadata/model_comparison.csv` dari notebook training).

In [ ]:
# Sub-Step 14.1
# Tujuan: Gabungkan seluruh xai_results jadi satu tabel ringkasan

summary_rows = []
for name, res in xai_results.items():
    row = {
        "model": name,
        "spatial_method": res.get("spatial_method", "GradCAM"),
        "centroid_vs_center_offset_r": res.get("centroid_vs_center_offset_r"),
        "centroid_vs_center_offset_p": res.get("centroid_vs_center_offset_p"),
        "probe_reposition_mean_delta": res.get("probe_reposition_mean_delta"),
        "probe_occlusion_center_mean_delta": res.get("probe_occlusion_center_mean_delta"),
        "probe_color_jitter_mean_delta": res.get("probe_color_jitter_mean_delta"),
        "nn_lineage_label_agreement_mean": res.get("nn_lineage_label_agreement_mean"),
        "head_agreement_heatmap_corr": res.get("head_agreement_heatmap_corr"),
        "gb_resnet_agreement_rate": res.get("gb_resnet_agreement_rate"),
        "resnet_correct_but_gb_disagrees_rate": res.get("resnet_correct_but_gb_disagrees_rate"),
    }
    # TCAV: model keluarga A/B simpan langsung di res["tcav"]; sub-model 07/08 (Section 10.5)
    # simpan terpisah per head (type vs damage) karena satu concept hanya valid di salah satu
    # head -- digabung di sini supaya kolom tcav_<konsep> seragam di SEMUA 10 model. PENTING:
    # kedua dict type/damage punya ke-4 key CONCEPT_DEFS sekaligus (None untuk yang N/A di
    # head itu), jadi {**a, **b} SALAH -- b akan menimpa nilai valid a dengan None. Harus
    # pilih yang bukan None per key, bukan unpacking dict biasa.
    def _merge_submodel_tcav(d_type, d_damage):
        return {k: (d_type.get(k) if d_type.get(k) is not None else d_damage.get(k))
                for k in set(d_type) | set(d_damage)}

    tcav_dict = res.get("tcav")
    if tcav_dict is None and "tcav_type_submodel" in res:
        tcav_dict = _merge_submodel_tcav(res["tcav_type_submodel"], res["tcav_damage_submodel"])
    elif tcav_dict is None and "tcav_type_head" in res:
        tcav_dict = _merge_submodel_tcav(res["tcav_type_head"], res["tcav_damage_head"])
    if tcav_dict:
        for cname, cval in tcav_dict.items():
            row[f"tcav_{cname}_score"] = cval["tcav_score"] if cval is not None else None  # None = N/A arsitektural
            row[f"tcav_{cname}_p"] = cval["p_value"] if cval is not None else None

    # GB tidak punya TCAV (lihat Section 9.5) -- peringkat SHAP per konsep adalah padanannya
    shap_rank = res.get("concept_equivalent_shap_rank")
    if shap_rank:
        for cname, per_class_rank in shap_rank.items():
            row[f"shap_rank_{cname}"] = per_class_rank

    summary_rows.append(row)

xai_summary_df = pd.DataFrame(summary_rows)
Path("metadata").mkdir(exist_ok=True)
xai_summary_df.to_csv("metadata/xai_summary.csv", index=False)

hyp2_df = pd.DataFrame(hypothesis2_results)
hyp2_df.to_csv("metadata/xai_hypothesis2_damage_leak.csv", index=False)
hyp3_df = pd.DataFrame(hypothesis3_results)
hyp3_df.to_csv("metadata/xai_hypothesis3_real_world.csv", index=False)

print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {'BELUM final (sampel kecil, hanya tes kode)' if DRY_RUN else 'hasil run penuh'}")
print()
pd.set_option("display.max_columns", None, "display.width", 200)
print("=== Ringkasan utama (hipotesis #1 posisi, #4 warna, #5 ukuran, TCAV) ===")
print(xai_summary_df.round(4).to_string(index=False))
print()
print("=== Hipotesis #2: kebocoran konsep 'rusak' ===")
print(hyp2_df.round(4).to_string(index=False))
print()
print("=== Hipotesis #3: generalisasi real_world ===")
print(hyp3_df.round(4).to_string(index=False))

In [ ]:
# Sub-Step 14.2
# Tujuan: Interpretasi ringkas -- dibaca manual bersama tabel di atas

print("""
Cara membaca tabel xai_summary.csv:
- centroid_vs_center_offset_r/p mendekati 0 (p tidak signifikan) di SEMUA model keluarga A/B
  -> konsisten dengan dugaan bahwa crop-to-bbox sudah menetralkan shortcut framing 'premium
  lebih terpusat' yang ditemukan EDA v2 -- variasi posisi yang tersisa di dataset_preprocessed
  sudah terlalu kecil untuk dieksploitasi model manapun.
- probe_reposition_mean_delta / probe_occlusion_center_mean_delta yang besar berarti model
  SENSITIF terhadap intervensi itu -- untuk occlusion_center, sensitivitas tinggi itu WAJAR
  (bean memang ada di tengah); untuk reposition, sensitivitas tinggi justru mengkhawatirkan
  (berarti posisi masih berpengaruh meski sudah di-crop+augmentasi).
- nn_lineage_label_agreement_mean rendah pada gambar yang salah klasifikasi mengindikasikan
  masalah DATA (near-duplicate cross-class, EDA v2 Section 09), bukan model.
- head_agreement_heatmap_corr tinggi di Model 08 (satu backbone) WAJAR karena representasi
  awal dibagi; kalau juga tinggi di Model 07 (dua model independen), itu sinyal spesialisasi
  yang diharapkan desain hierarchical TIDAK benar-benar terjadi secara spasial.
- resnet_correct_but_gb_disagrees_rate yang tinggi (Model 10) mengkonfirmasi kenapa ensemble
  gagal di laporan training: gradient boosting sering menarik ke arah salah justru di kasus
  yang CNN sudah benar.
- probe_color_jitter_mean_delta & tcav_dark_color_score/p (hipotesis #4 -- BARU): warna
  adalah fitur EDA dengan effect size terbesar, belum pernah diuji sebelumnya. Delta besar
  di luar rentang augmentasi training (brightness/saturation training cuma +-10%/+-5%)
  berarti model rapuh terhadap variasi warna yang lebih ekstrem dari yang pernah dilatih.
- tcav_large_area_score/p (hipotesis #5 -- BARU): sama seperti off_center/elongated_shape,
  p tidak signifikan berarti belum cukup bukti ukuran-di-frame dipakai sebagai shortcut.
- Baris 01_gradient_boosting: kolom tcav_* kosong (NaN) karena GB memang tidak punya CAV --
  lihat kolom shap_rank_* sebagai padanannya (peringkat 1 = fitur paling penting SHAP untuk
  kelas itu; peringkat kecil pada fitur yang bersesuaian = indikasi konsep itu jadi sinyal
  utama GB, sama seperti tcav_score tinggi & signifikan pada model neural).
- Baris 07_hierarchical/08_multitask: kolom tcav_* di sini gabungan dari head type & damage
  (tiap konsep valid di salah satu, lihat Section 10.5) -- NaN pada sel tertentu berarti
  target kelas konsep itu memang TIDAK ADA di ruang output sub-model manapun (N/A arsitektural,
  bukan celah pengujian).
- xai_hypothesis2_damage_leak.csv (hipotesis #2 -- BARU): damage_leak_score tinggi ARTINYA
  banyak gambar non-defect yang sensitif ke arah 'rusak'; damage_leak_vs_misclass_r positif
  & signifikan berarti sensitivitas itu BUKAN cuma angka abstrak -- benar-benar berkorelasi
  dengan kesalahan klasifikasi ke defect (konfirmasi kuantitatif hipotesis EDA v3).
- xai_hypothesis3_real_world.csv (hipotesis #3 -- BARU): real_world_confidence_mean jauh
  lebih rendah dari test_confidence_mean, atau distribusi kelas prediksinya sangat timpang
  dibanding test, adalah sinyal model kurang generalisasi ke kondisi pengambilan gambar
  yang berbeda -- meski lolos semua verifikasi di atas untuk data sekelas test.

Lihat docs/xai-strategy.md untuk alasan lengkap tiap metode, dan diskusikan hasil BELUM final
(DRY_RUN) ini sebelum full run -- terutama urutan prioritas: perbaiki dulu apa pun yang
XAI temukan di sini, baru lanjut ke validasi 4-fold CV.
""")

## Section 15 — Visualisasi untuk Laporan HTML

Seluruh section sebelumnya cuma mencetak angka. Section ini menghasilkan gambar (Grad-CAM/IG/
Occlusion overlay, contoh probe kausal, dua heatmap head berdampingan) yang disimpan sebagai
PNG ke `results/xai_figures/` -- dipakai laporan HTML, bukan bagian dari analisis statistik
(N_VIZ_PER_CLASS tetap kecil terlepas dari DRY_RUN, karena tujuannya cuma ilustrasi visual,
bukan agregat). Model yang dipakai tetap checkpoint yang sama dari Section 5 -- tidak ada
pelatihan ulang atau perubahan angka statistik di section manapun sebelumnya.

In [ ]:
# Sub-Step 15.1
# Tujuan: Helper overlay heatmap (jet colormap) di atas gambar asli

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

FIG_DIR = Path("results/xai_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_VIZ_PER_CLASS = 1


def heatmap_overlay_rgb(pil_img, heatmap, alpha=0.5):
    img_arr = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")) / 255.0
    hm = heatmap - np.nanmin(heatmap)
    hm = hm / (np.nanmax(hm) + 1e-8)
    heat_colored = cm.jet(hm)[:, :, :3]
    overlay = (1 - alpha) * img_arr + alpha * heat_colored
    return np.clip(overlay, 0, 1)

In [ ]:
# Sub-Step 15.2
# Tujuan: Grad-CAM + IG + Occlusion overlay -- 09_noise_robust (juara) & 05_convnext_tiny (pembanding)

viz_df = sample_per_class(test_df, N_VIZ_PER_CLASS)
viz_imgs = load_pil_batch(viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
viz_labels = np.array([LABEL_TO_IDX[l] for l in viz_df["label"]])
viz_pil_originals = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in viz_df["image_path"]]

for model_name in ["09_noise_robust", "05_convnext_tiny"]:
    model = cnn_models[model_name]
    layer = FAMILY_A_LAYER[model_name](model)
    gc_maps = gradcam_heatmaps(model, layer, viz_imgs, viz_labels)
    ig_maps_v = ig_heatmaps(model, viz_imgs, viz_labels)
    occ_maps_v = occlusion_heatmaps(model, viz_imgs, viz_labels)

    n = len(viz_df)
    fig, axes = plt.subplots(n, 4, figsize=(10, 2.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        axes[i, 0].imshow(viz_pil_originals[i]); axes[i, 0].set_ylabel(viz_df.iloc[i]["label"], fontsize=10)
        axes[i, 1].imshow(heatmap_overlay_rgb(viz_pil_originals[i], gc_maps[i]))
        axes[i, 2].imshow(heatmap_overlay_rgb(viz_pil_originals[i], ig_maps_v[i]))
        axes[i, 3].imshow(heatmap_overlay_rgb(viz_pil_originals[i], occ_maps_v[i]))
        for j in range(4):
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if i == 0:
            for j, t in enumerate(["Original", "Grad-CAM", "Integrated Gradients", "Occlusion"]):
                axes[i, j].set_title(t, fontsize=10)
    fig.suptitle(model_name, fontsize=12)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"gradcam_{model_name}.png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"Disimpan: {FIG_DIR / f'gradcam_{model_name}.png'}")

In [ ]:
# Sub-Step 15.3
# Tujuan: IG + Occlusion overlay -- 06_deit_tiny (tanpa Grad-CAM, tidak ada conv layer)

model = cnn_models["06_deit_tiny"]
ig_maps_deit = ig_heatmaps(model, viz_imgs, viz_labels)
occ_maps_deit = occlusion_heatmaps(model, viz_imgs, viz_labels)

n = len(viz_df)
fig, axes = plt.subplots(n, 3, figsize=(7.5, 2.5 * n))
if n == 1:
    axes = axes.reshape(1, -1)
for i in range(n):
    axes[i, 0].imshow(viz_pil_originals[i]); axes[i, 0].set_ylabel(viz_df.iloc[i]["label"], fontsize=10)
    axes[i, 1].imshow(heatmap_overlay_rgb(viz_pil_originals[i], ig_maps_deit[i]))
    axes[i, 2].imshow(heatmap_overlay_rgb(viz_pil_originals[i], occ_maps_deit[i]))
    for j in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    if i == 0:
        for j, t in enumerate(["Original", "Integrated Gradients", "Occlusion"]):
            axes[i, j].set_title(t, fontsize=10)
fig.suptitle("06_deit_tiny", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "gradcam_06_deit_tiny.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'gradcam_06_deit_tiny.png'}")

In [ ]:
# Sub-Step 15.4
# Tujuan: Contoh probe kausal: reposisi & occlusion (before/after + delta probabilitas)

probe_examples = []
for _, row in viz_df.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    predict_fn_09 = lambda img: predict_proba_pil(cnn_models["09_noise_robust"], img)
    crop_a, crop_b = probe_reposition(RAW_DIR / row["orig_path"])
    d_rep, pa_rep, pb_rep = prob_delta(predict_fn_09, crop_a, crop_b, cls_idx)
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d_occ, pa_occ, pb_occ = prob_delta(predict_fn_09, prep_img, occluded, cls_idx)
    probe_examples.append((row["label"], crop_a, crop_b, pa_rep, pb_rep, prep_img, occluded, pa_occ, pb_occ))

n = len(probe_examples)
fig, axes = plt.subplots(n, 4, figsize=(10, 2.6 * n))
if n == 1:
    axes = axes.reshape(1, -1)
for i, (label, crop_a, crop_b, pa_rep, pb_rep, prep_img, occluded, pa_occ, pb_occ) in enumerate(probe_examples):
    axes[i, 0].imshow(crop_a); axes[i, 0].set_title(f"asli\nP({label})={pa_rep:.2f}", fontsize=9)
    axes[i, 1].imshow(crop_b); axes[i, 1].set_title(f"digeser\nP({label})={pb_rep:.2f}", fontsize=9)
    axes[i, 2].imshow(prep_img); axes[i, 2].set_title(f"asli\nP({label})={pa_occ:.2f}", fontsize=9)
    axes[i, 3].imshow(occluded); axes[i, 3].set_title(f"occluded\nP({label})={pb_occ:.2f}", fontsize=9)
    axes[i, 0].set_ylabel(label, fontsize=10)
    for j in range(4):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
fig.suptitle("Probe kausal (09_noise_robust): reposisi (kiri) vs occlusion tengah (kanan)", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "probe_examples.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'probe_examples.png'}")

In [ ]:
# Sub-Step 15.5
# Tujuan: Head-agreement visual: 2 heatmap (tipe vs rusak) berdampingan -- Model 07 & 08

viz_nd = sample_per_class(test_df[test_df["label"] != "defect"], N_VIZ_PER_CLASS)
viz_nd_imgs = load_pil_batch(viz_nd["image_path"].tolist(), PREP_DIR, eval_transform)
viz_nd_pil = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in viz_nd["image_path"]]
viz_nd_type_targets = np.array([TYPE_LABEL_FN(l) for l in viz_nd["label"]])
viz_nd_damage_targets = np.zeros(len(viz_nd), dtype=int)

gc_type_07 = gradcam_heatmaps(type_model, type_model.features[-1], viz_nd_imgs, viz_nd_type_targets)
gc_damage_07 = gradcam_heatmaps(damage_model, damage_model.features[-1], viz_nd_imgs, viz_nd_damage_targets)
gc_damage_08 = gradcam_heatmaps(damage_wrapper, mt_layer, viz_nd_imgs, viz_nd_damage_targets)
gc_type_08 = gradcam_heatmaps(type_wrapper, mt_layer, viz_nd_imgs, viz_nd_type_targets)

n = len(viz_nd)
fig, axes = plt.subplots(n, 5, figsize=(12.5, 2.6 * n))
if n == 1:
    axes = axes.reshape(1, -1)
for i in range(n):
    axes[i, 0].imshow(viz_nd_pil[i])
    axes[i, 1].imshow(heatmap_overlay_rgb(viz_nd_pil[i], gc_type_07[i]))
    axes[i, 2].imshow(heatmap_overlay_rgb(viz_nd_pil[i], gc_damage_07[i]))
    axes[i, 3].imshow(heatmap_overlay_rgb(viz_nd_pil[i], gc_type_08[i]))
    axes[i, 4].imshow(heatmap_overlay_rgb(viz_nd_pil[i], gc_damage_08[i]))
    axes[i, 0].set_ylabel(viz_nd.iloc[i]["label"], fontsize=10)
    for j in range(5):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    if i == 0:
        for j, t in enumerate(["Original", "07: head-tipe", "07: head-rusak", "08: head-tipe", "08: head-rusak"]):
            axes[i, j].set_title(t, fontsize=9)
fig.suptitle("Head-agreement: 07 Hierarchical (2 model terpisah) vs 08 Multi-task (1 backbone)", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "head_agreement.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'head_agreement.png'}")
print("Semua figure Section 15 selesai disimpan di results/xai_figures/")